# 关于模型 invoke 的使用

## 1. invoke 的传参

### 1.1 文本信息

In [2]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model(
    model='deepseek-flash'
)

/Users/refone/Coding/ai/langchain-learn/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [3]:
res = model.invoke("翻译如下汉子:你好世界")
print(res)

content='Hello World' additional_kwargs={'refusal': None, 'reasoning_content': 'The user wants me to translate the Chinese phrase "你好世界" into English. This is a straightforward translation task with clear source and target languages indicated.\n\n"你好世界" is the classic phrase used in programming examples, equivalent to "Hello World." It\'s a simple greeting combined with the word "world." The translation is direct and unambiguous. No special terminology or cultural nuances are involved here—just a standard greeting phrase that has become iconic in programming contexts.\n\nI will provide the English translation directly.'} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 36, 'total_tokens': 138, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 99, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'i

### 1.2 字典列表

- 角色说明
  - `system` 设定 AI 的行为、角色, 能稍稍影响模型的 MoE 能力
  - `user` 用户输入
  - `assistant` AI 的历史恢复(用于对话上下文)

- 大模型本身没有记忆

In [7]:
messages = [
    { "role":"system", "content":"你是一个专业的数学老师" },
    { "role":"user", "content":"1 + 2 = ?" },
    { "role":"assistant", "content":"3" },
    { "role":"user", "content":"我刚才问了什么问题?" }
]

res = model.invoke(messages)
print(res)

content='你刚才问的是：**“1 + 2 = ?”**' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文：“我刚才问了什么问题?” 用户之前问的是“1 + 2 = ?”，我回答3。现在问刚才问了什么问题。需要直接回答。注意可能用户是在测试记忆。应回答：你刚才问的是“1 + 2 = ?”。也可以用中文。确保准确。'} response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 50, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 64, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 50}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '31c68ff5-3421-4029-9578-c97e86373560', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0a5bb-3f9f-7213-9357-163c6d9645eb-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'inp

In [3]:
conversation = [
    {"role": "system", "content": "你是一个非常友好的 AI 助手"},
    {"role":"user", "content": "你好, 我叫小明"},
]

# 第 1 次对话
response1 = model.invoke(conversation)
print(f"AI 的回复 1: {response1.content}")

# 添加记忆
conversation.append({"role":"assistant", "content":response1.content})
conversation.append({"role":"user", "content":"我叫什么名字?"})

# 第 2 次对话
response2 = model.invoke(conversation)
print(f"AI 的回复 2: {response2.content}")

AI 的回复 1: 你好呀，小明！很高兴认识你 😊  
我是你的 AI 助手，有什么想聊的、想问的，或者需要帮忙的，都可以告诉我～
AI 的回复 2: 你叫小明呀 😊  
刚才你告诉我啦～


### 1.3 消息对象列表

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content='你是一个问什么答什么的数学老师'),
    HumanMessage(content='给出圆面积计算公式')
]
# 第 1 次对话
response1 = model.invoke(messages)
print(f"AI 回复 1: {response1.content}")

# 添加记忆
messages.append(AIMessage(content=response1.content))
messages.append(HumanMessage(content="我刚才问了什么问题?"))

# 第 2 次对话
response2 = model.invoke(messages)
print(f"AI 回复 2: {response2.content}")

AI 回复 1: 圆的面积公式：

\[
S=\pi r^2
\]

其中：

- \(S\) 表示圆的面积；
- \(r\) 表示圆的半径；
- \(\pi\) 是圆周率，通常取 \(3.14\) 或 \(\frac{22}{7}\)。

如果已知直径 \(d\)，因为 \(r=\frac{d}{2}\)，所以也可以写成：

\[
S=\pi\left(\frac{d}{2}\right)^2=\frac{\pi d^2}{4}
\]
AI 回复 2: 你刚才问的是：

**“给出圆面积计算公式。”**


## 2. AIMessage 字段解读

`model.invoke` 返回的是 `AIMessage` 信息, 其中除了 content, 还有很多其他字段.

In [9]:
response = model.invoke([HumanMessage(content='2 + 3 * 2 = ?')])

In [10]:
type(response)

langchain_core.messages.ai.AIMessage

In [11]:
from rich import print as rprint

rprint(response)

AIMessage(
    content='8',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'We need answer arithmetic order operations. Need answer concise. 2+3*2 = 2+6=8. Need 
maybe mention multiplication first. final.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 35,
            'prompt_tokens': 39,
            'total_tokens': 74,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 33,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 39
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '5523e57d-3f26-4d3a-ae02-5fa7f80417b7',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0a7e9-306c-7ef3-8017-0212e821455e-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 39,
        'output_tokens': 35,
        'total_tokens': 74,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 33}
    }
)

In [12]:
response = model.invoke("用一句话解释什么是 AI")

# 1. 获取回复内容
print("AI 回复:", response.content)

# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")

# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")

# 4. 获取消息 ID
print(f"消息 ID: {response.id}")

AI 回复: AI（人工智能）是让机器通过算法和数据模拟、延伸人类智能，从而完成感知、学习、推理、决策和生成等任务的技术。
使用的模型: deepseek-flash
结束原因: stop
模型提供商：deepseek

输入 tokens: 35
输出 tokens: 94
总计 tokens: 129
消息 ID: lc_run--01a0a7ea-9915-72c0-af9f-e0f660117358-0
